In [1]:
import pandas as pd
import pyam
from NFormat import NFormat

<IPython.core.display.Javascript object>

## Data input
### The following gathers demand data from various sources into dataframes

#### Reading in data from [Canada's Energy Future](https://apps.cer-rec.gc.ca/ftrppndc/dflt.aspx?GoCTemplateCulture=en-CA)

This data will be used for specific residential and commerical energy demand, as well as transport demand

In [2]:
energy_future_data = NFormat('data/can_energy_future.xlsx', 'can_energy_future')

#### Reading in data from [Statistics Canada](https://www150.statcan.gc.ca/t1/tbl1/en/cv.action?pid=2510002901)

This data will be used for industrial feedstock demand

In [3]:
i_feed = NFormat('data/statcan_data.csv', 'statcan')

#### Reading in data from [Natural Resources Canada](https://oee.nrcan.gc.ca/corporate/statistics/neud/dpa/menus/trends/comprehensive_tables/list.cfm)

This data will be used for residential and commerical thermal demand before 2020 (might not be needed?)

In [12]:
space_heating = NFormat('data/space_heating.xlsx', 'space_heating')

# Downscaling

In [6]:
# Data from previous msg run
msg_data = pd.read_excel('~/message-ix/data/MESSAGE_CA__test__v4.xlsx', sheet_name = 'demand')
msg_data = msg_data[msg_data['year'] >= 2020]
msg_data = msg_data[['year', 'node', 'commodity', 'unit', 'value']]
msg_data = pyam.IamDataFrame(msg_data, model = 'MESSAGEix', scenario = 'None', variable = 'commodity', region = 'node')

In [13]:
# Downscale space heating from territories to NWT, Nunavut, Yukon
territories = space_heating.filter(region = 'Territories')
territories.append(energy_future_data.filter(region = ['Northwest Territories', 'Yukon', 'Nunavut']), inplace = True)
new = territories.downscale_region('Space Heating Usage', region = 'Territories', proxy = 'rc_spec')
space_heating.append(new, inplace = True)
space_heating = space_heating.filter(region = energy_future_data.region)

In [14]:
# Thermal downscaling
therm = msg_data.append(energy_future_data)
i_therm = therm.downscale_region('i_therm', region = 'Canada', proxy = 'i_spec')
rc_therm = therm.downscale_region('rc_therm', region = 'Canada', proxy = 'rc_spec')

In [15]:
# Feedstock downscaling
feed = msg_data.append(energy_future_data)
i_feed_proj = feed.downscale_region('i_feed', region = 'Canada', proxy = 'i_spec')

In [16]:
msg_years = list(range(2020, 2061, 5))

total = energy_future_data
total.append(i_feed, inplace = True)
total.append(space_heating, inplace = True)
total.append(i_therm, inplace = True)
total.append(rc_therm, inplace = True)
total.append(i_feed_proj, inplace = True)
total.append(msg_data.filter(year = msg_years, variable = ['i_feed', 'i_therm', 'rc_therm']), inplace = True)
final = total.filter(year = [msg_years])

In [17]:
final = final.convert_unit('TJ', to = 'GWh').convert_unit('GWh', to = 'GWa', factor = 1/8760)
final = final.convert_unit('PJ', to = 'GWh').convert_unit('GWh', to = 'GWa', factor = 1/8760)

In [18]:
# This should write to the data folder of the repo (will need to be change to work with other computers)
final.to_excel('~/message-ix/data/canadian_energy_demand.xlsx')

# Below needs to be tested

In [ ]:
test = final.filter(year = [2055, 2060])

In [ ]:
# Figure out way to get 2055/2060 values
regions = spec.region
regions.remove('Canada')
incl = []

ratios = spec.filter(variable = 'rc_spec')
for region in regions:
    incl.append(region + '/Canada')
    ratios.divide(region, 'Canada', region + '/Canada', axis = 'region', append = True)


In [ ]:
ratios_pd = ratios.filter(region = incl).as_pandas()

In [ ]:
ratios_pd

In [ ]:
ratios_pd = ratios_pd[['region', 'year', 'value']]

In [ ]:
ratios_pd[ratios_pd['region'] == 'Alberta/Canada']